# Construction du dataset — Segmentation YOLO

**Projet** : Gallica Images — Classification gravures sur bois vs cuivre  
**Date**   : Avril 2026

Ce notebook construit le dataset d'images segmentées pour entraîner
le classifieur bois/cuivre. Il télécharge les pages depuis les sources
IIIF et l'API BnF, puis extrait les illustrations avec YOLO.

---

## Sources bois

| Dossier | ARK / Source | Lieu & Date | Graveur |
|---|---|---|---|
| `bois_salomon_lyon1557` | `btv1b2200047r` — API BnF | Lyon, 1557 | Bernard Salomon |
| `bois_wickram_mayence1545` | `bsb10139926` — BSB Munich IIIF | Mayence, 1545 | Jörg Wickram (1505?–1560?) |
| `bois_solis_francfort1581` | `bsb00087854` — BSB Munich IIIF | Francfort-sur-le-Main, 1581 | Virgil Solis |

## Sources cuivre

| Dossier | ARK / Source | Lieu & Date | Graveur |
|---|---|---|---|
| `cuivre_clein_paris1637` | `bsb10863401` — BSB Munich IIIF | Paris, 1637 | Francisco Clein (inv.) & Salomon Savery (sculp.) |
| `cuivre_passe_metamorphoseon` | `bpt6k15218623` — PDF Gallica | — | Crispin de Passe |
| `cuivre_passe_nasonis` | `bpt6k1522448r` — PDF Gallica | — | Crispin de Passe |
| `cuivre_renouard_traduites` | `bpt6k6277348n` — PDF Gallica | — | non renseigné |
| `cuivre_renouard_traduittes` | `bpt6k722055` — PDF Gallica | — | non renseigné |

---

## Data augmentation

Pour enrichir le dataset d'entraînement, chaque illustration est complétée
par sa version **flippée horizontalement**. Le flip est appliqué uniquement
sur le split `train` — les splits `val` et `test` restent inchangés
pour garantir une évaluation propre.

---

⚠️ **Note mémoire GPU** : YOLO est chargé pour la segmentation puis libéré
avant le split du dataset. Ne pas charger ResNet50 dans ce notebook.

---

## 1. Configuration

In [1]:
import sys
sys.path.insert(0, "..")
from gallica_utils import (
    charger_yolo, segmenter_page, segmenter_corpus,
    liberer_yolo, telecharger_pages_iiif,
    rassembler_illustrations, split_dataset, stats_illustrations,
    BASE_URL, ARK_SALOMON
)

import os
import shutil
import requests
import subprocess
import torch
from PIL import Image, ImageOps
from io import BytesIO

# ── Chemins ─────────────────────────────────────────────────
DOSSIER_IMAGES_BRUTES = "../../../data/sources"
DOSSIER_SEGMENTEES    = "../../../data/segmentees"
DOSSIER_DATASET       = "../../../data/datasets/bois_cuivre"
DOSSIER_PDF           = "../../../data/pdf_cuivre"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

Device : cuda


## 2. Chargement de YOLOv5

Modèle `seglinglin/Historical-Illustration-Extraction` depuis Hugging Face.  
Entraîné sur des documents historiques — détecte les illustrations dans les pages imprimées.

In [16]:
# Cloner YOLOv5 si absent
if not os.path.exists("../../../yolov5_repo"):
    subprocess.run(["git", "clone",
                    "https://github.com/ultralytics/yolov5.git",
                    "../../../yolov5_repo"])
    print("✓ YOLOv5 cloné")

modele_yolo = charger_yolo(yolov5_repo="../../../yolov5_repo")

✓ YOLO chargé — classes : {0: 'illustration'}


## 3. Classe bois

### 3.1 Bernard Salomon — Lyon 1557 — API BnF

##### <u>Titre :</u> Excellente figueren ghesneden vuyten vppersten Poëte Ouidius vuyt vyfthien boucken der veranderinghe met huerlier bedietsele. Duer Guilliaume Borluit burgher der stede van Ghendt.

##### <u>Graveur :</u> Salomon, Bernard

##### <u>ARK :</u> `btv1b2200047r` — Source : API BnF (`/api/ouvrages/{ark}/illustrations`)

##### <u>Dossier :</u> `bois_salomon_lyon1557/`

In [17]:
r             = requests.get(f"{BASE_URL}/api/ouvrages/{ARK_SALOMON}/illustrations", timeout=30)
illustrations = r.json()
valides       = [
    illus for illus in illustrations
    if illus.get("metas", {}).get("content_embedding")
    and len(illus["metas"]["content_embedding"]) == 768
]
print(f"Illustrations Salomon avec embedding : {len(valides)}")

Illustrations Salomon avec embedding : 184


In [18]:
dossier_brut_salomon = f"{DOSSIER_IMAGES_BRUTES}/bois_salomon_lyon1557"
os.makedirs(dossier_brut_salomon, exist_ok=True)
pages_bois_brutes = []

for i, illus in enumerate(valides):
    print(f"  {i+1}/{len(valides)}...", end="\r")
    metas  = illus.get("metas", {})
    url    = metas.get("link", "")
    view   = illus.get("view_number", i)
    chemin = f"{dossier_brut_salomon}/salomon_f{view:03d}.jpg"

    if os.path.exists(chemin):
        pages_bois_brutes.append(chemin)
        continue
    try:
        r   = requests.get(url, timeout=15)
        img = Image.open(BytesIO(r.content)).convert("RGB")
        img.save(chemin)
        pages_bois_brutes.append(chemin)
    except Exception as e:
        print(f"\n  Erreur page {view} : {e}")

print(f"\n✓ {len(pages_bois_brutes)} pages Salomon sauvegardées")

  184/184...
✓ 184 pages Salomon sauvegardées


In [19]:
segmenter_corpus(
    pages_bois_brutes,
    f"{DOSSIER_SEGMENTEES}/bois_salomon_lyon1557",
    modele_yolo,
    conf_thres=0.25
)

  184/184...
✓ 204 illustrations extraites dans ../../../data/segmentees/bois_salomon_lyon1557


204

### 3.2 Jörg Wickram — Mayence 1545 — BSB Munich `bsb10139926`

##### <u>Titre :</u> P. Ouidij Nasonis dess aller sinnreichsten Poeten Metamorphosis / Das ist : von der wunderbarlicher Verenderung der Gestalten der Menschen / Thiern vnd anderer Creaturen etc. Jedermann lüstlich / besonders aber allen Malern / Bildthauwern / vnnd dergleichen Kunstlern nützlich zu gebrauchen.

##### <u>Graveur :</u> Wickram, Jörg (1505?–1560?)

##### <u>ARK :</u> `bsb10139926` — Source : BSB Munich IIIF

##### <u>Dossier :</u> `bois_wickram_mayence1545/`

In [20]:
pages_mayence = telecharger_pages_iiif(
    "https://api.digitale-sammlungen.de/iiif/presentation/v2/bsb10139926/manifest",
    f"{DOSSIER_IMAGES_BRUTES}/bois_wickram_mayence1545",
    prefixe="wickram"
)

segmenter_corpus(
    pages_mayence,
    f"{DOSSIER_SEGMENTEES}/bois_wickram_mayence1545",
    modele_yolo,
    conf_thres=0.25
)

Pages trouvées : 332 — Ovidius Naso, Publius: P. Ouidij Nasonis deß aller sinnreich
  332/332...
✓ 332 pages sauvegardées dans ../../../data/sources/bois_wickram_mayence1545
  332/332...
✓ 70 illustrations extraites dans ../../../data/segmentees/bois_wickram_mayence1545


70

### 3.3 Virgil Solis — Francfort-sur-le-Main 1581 — BSB `bsb00087854`

##### <u>Titre :</u> P. Ovidii Metamorphosis, Oder : Wunderbarliche vnnd seltzame Beschreibung / von der Menschen / Thiern / vnnd anderer Creaturen Veränderung auch von dem Wandeln / Leben vnd Thaten der Götter / Martis / Veneris / Mercurij / etc. Allen Poeten / Malern / Goldschmiden / Bildthauwern / vnnd Liebhabern der edlen Poesi vnd fürnembsten Künsten / Nützlich vnd lustig zu lesen / Jetzt widerum auff ein newes / dem gemeinen Vatterlande Teutscher Sprach zu grossem nutz vnd dienst auss sonderlichem fleiss mit schönen Figurn renouiert / corrigiert / vnd an Tag geben / durch Sigmund Feyerabendt Buchhändlern.

##### <u>Graveur :</u> Solis, Virgil

##### <u>ARK :</u> `bsb00087854` — Source : BSB Munich IIIF

##### <u>Dossier :</u> `bois_solis_francfort1581/`

In [21]:
pages_bsb87854 = telecharger_pages_iiif(
    "https://api.digitale-sammlungen.de/iiif/presentation/v2/bsb00087854/manifest",
    f"{DOSSIER_IMAGES_BRUTES}/bois_solis_francfort1581",
    prefixe="solis"
)

segmenter_corpus(
    pages_bsb87854,
    f"{DOSSIER_SEGMENTEES}/bois_solis_francfort1581",
    modele_yolo,
    conf_thres=0.25
)

Pages trouvées : 435 — Ovidius Naso, Publius: P. Ovidii Metamorphosis, Oder: Wunder
  435/435...
✓ 435 pages sauvegardées dans ../../../data/sources/bois_solis_francfort1581
  435/435...
✓ 199 illustrations extraites dans ../../../data/segmentees/bois_solis_francfort1581


199

## 4. Classe cuivre

### 4.1 Clein & Savery — Paris 1637 — BSB `bsb10863401`

##### <u>Titre :</u> Pub. Ovidii Nasonis Metamorphoseon libri XV, ad fidem editionum optimarum et codicum manuscriptorum examinati, animadversi, necnon notis illustrati, opera et studio Thomae Farnabii. Editio nunc primum in Gallia et multis figuris aeneis adornata.

##### <u>Graveur :</u> Clein, Francisco (inv.) et Savery, Salomon (sculp.)

##### <u>ARK :</u> `bsb10863401` — Source : BSB Munich IIIF

##### <u>Dossier :</u> `cuivre_clein_paris1637/`

In [22]:
pages_munich = telecharger_pages_iiif(
    "https://api.digitale-sammlungen.de/iiif/presentation/v2/bsb10863401/manifest",
    f"{DOSSIER_IMAGES_BRUTES}/cuivre_clein_paris1637",
    prefixe="clein"
)

segmenter_corpus(
    pages_munich,
    f"{DOSSIER_SEGMENTEES}/cuivre_clein_paris1637",
    modele_yolo,
    conf_thres=0.25
)

Pages trouvées : 138 — Ovidius Naso, Publius: Metamorphoseon libri XV.
  138/138...
✓ 138 pages sauvegardées dans ../../../data/sources/cuivre_clein_paris1637
  138/138...
✓ 25 illustrations extraites dans ../../../data/segmentees/cuivre_clein_paris1637


25

### 4.2 PDFs Gallica — 4 éditions cuivre

| Dossier | ARK | Titre abrégé | Graveur |
|---|---|---|---|
| `cuivre_passe_metamorphoseon` | `bpt6k15218623` | Metamorphoseon Ovidianarum typi aliquot... | Crispin de Passe |
| `cuivre_passe_nasonis` | `bpt6k1522448r` | P. Ovid. Nasonis XV Metamorphoseon librorum figurae... | Crispin de Passe |
| `cuivre_renouard_traduites` | `bpt6k6277348n` | Les Métamorphoses d'Ovide, traduites en prose françoise... | non renseigné |
| `cuivre_renouard_traduittes` | `bpt6k722055` | Les métamorphoses d'Ovide, traduittes en prose françoise... | non renseigné |

In [23]:
import fitz  # PyMuPDF — pip install pymupdf

# Correspondance nom de fichier PDF → nom de dossier lisible
NOM_DOSSIERS_PDF = {
    "bpt6k15218623" : "cuivre_passe_metamorphoseon",
    "bpt6k1522448r" : "cuivre_passe_nasonis",
    "bpt6k6277348n" : "cuivre_renouard_traduites",
    "bpt6k722055"   : "cuivre_renouard_traduittes",
}

def extraire_pages_pdf(chemin_pdf, dossier_sortie, dpi=150):
    """Convertit chaque page d'un PDF en JPG."""
    os.makedirs(dossier_sortie, exist_ok=True)
    doc   = fitz.open(chemin_pdf)
    pages = []
    for i, page in enumerate(doc):
        mat    = fitz.Matrix(dpi/72, dpi/72)
        pix    = page.get_pixmap(matrix=mat)
        chemin = f"{dossier_sortie}/page{i+1:03d}.jpg"
        pix.save(chemin)
        pages.append(chemin)
        print(f"  {i+1}/{len(doc)}...", end="\r")
    print(f"\n✓ {len(pages)} pages extraites depuis {os.path.basename(chemin_pdf)}")
    return pages

def trouver_nom_dossier(nom_fichier):
    """Trouve le nom de dossier lisible depuis le nom du fichier PDF."""
    for ark, nom in NOM_DOSSIERS_PDF.items():
        if ark in nom_fichier:
            return nom
    return os.path.splitext(nom_fichier)[0][:50]

# Traiter les 4 PDFs
toutes_pages_pdf = []
for fichier in sorted(os.listdir(DOSSIER_PDF)):
    if fichier.endswith(".pdf"):
        nom_dossier = trouver_nom_dossier(fichier)
        dossier     = f"{DOSSIER_IMAGES_BRUTES}/cuivre_pdf/{nom_dossier}"
        print(f"\nTraitement : {fichier[:60]}")
        print(f"  → dossier : {nom_dossier}")
        pages = extraire_pages_pdf(f"{DOSSIER_PDF}/{fichier}", dossier)
        toutes_pages_pdf.append((nom_dossier, pages))

print(f"\n✓ Total pages PDF : {sum(len(p) for _, p in toutes_pages_pdf)}")


Traitement : Les_métamorphoses_d'Ovide_traduittes_en_[...]Ovide_(0043_bpt
  → dossier : cuivre_renouard_traduittes
  746/746...
✓ 746 pages extraites depuis Les_métamorphoses_d'Ovide_traduittes_en_[...]Ovide_(0043_bpt6k722055.pdf

Traitement : Metamorphoseon_Ovidianarum_typi_aliquot_artificiosissimè_[..
  → dossier : cuivre_passe_metamorphoseon
  313/313...
✓ 313 pages extraites depuis Metamorphoseon_Ovidianarum_typi_aliquot_artificiosissimè_[...]Ovide_(0043_bpt6k15218623.pdf

Traitement : P_Ovid_Nasonis_XV_Metamorphoseon_[...]Salsmann_Wilhelm_bpt6k
  → dossier : cuivre_passe_nasonis
  284/284...
✓ 284 pages extraites depuis P_Ovid_Nasonis_XV_Metamorphoseon_[...]Salsmann_Wilhelm_bpt6k1522448r.pdf

Traitement : [Les_Métamorphoses_d'Ovide_traduites_en_[...]Ovide_(0043_bpt
  → dossier : cuivre_renouard_traduites
  1226/1226...
✓ 1226 pages extraites depuis [Les_Métamorphoses_d'Ovide_traduites_en_[...]Ovide_(0043_bpt6k6277348n.pdf

✓ Total pages PDF : 2569


In [24]:
# Segmenter chaque édition dans son propre sous-dossier
for nom_dossier, pages in toutes_pages_pdf:
    dossier_sortie = f"{DOSSIER_SEGMENTEES}/cuivre_pdf/{nom_dossier}"
    print(f"\n{nom_dossier}")
    segmenter_corpus(pages, dossier_sortie, modele_yolo, conf_thres=0.25)


cuivre_renouard_traduittes
  746/746...
✓ 30 illustrations extraites dans ../../../data/segmentees/cuivre_pdf/cuivre_renouard_traduittes

cuivre_passe_metamorphoseon
  313/313...
✓ 143 illustrations extraites dans ../../../data/segmentees/cuivre_pdf/cuivre_passe_metamorphoseon

cuivre_passe_nasonis
  284/284...
✓ 148 illustrations extraites dans ../../../data/segmentees/cuivre_pdf/cuivre_passe_nasonis

cuivre_renouard_traduites
  1226/1226...
✓ 36 illustrations extraites dans ../../../data/segmentees/cuivre_pdf/cuivre_renouard_traduites


## 5. Libérer la mémoire GPU

⚠️ Obligatoire avant de lancer le fine-tuning ResNet50 dans le notebook 05.

In [25]:
modele_yolo = liberer_yolo(modele_yolo)

✓ Mémoire GPU libérée


## 6. Récapitulatif des illustrations segmentées


Nettoyage des données à la main à cette étape;

In [32]:
stats_illustrations(DOSSIER_SEGMENTEES)

Illustrations segmentées :

── BOIS ──
  bois_salomon_lyon1557                              : 192
  bois_solis_francfort1581                           : 187
  bois_wickram_mayence1545                           : 52

── CUIVRE ──
  cuivre_clein_paris1637                             : 18
  cuivre_passe_metamorphoseon                        : 135
  cuivre_passe_nasonis                               : 136
  cuivre_renouard_traduites                          : 16
  cuivre_renouard_traduittes                         : 17

────────────────────────────────────────────────────────────
  Total bois   : 431
  Total cuivre : 322
  TOTAL        : 753


## 7. Data augmentation — flip horizontal

Pour chaque illustration segmentée, on génère une version flippée horizontalement.
Cela double la taille effective du dataset d'entraînement sans nouvelles sources.

Le flip est appliqué **sur toutes les illustrations segmentées** avant le split —
le split 70/15/15 garantit que les versions originale et flippée d'une même image
ne se retrouvent pas dans des splits différents.

In [33]:
def flipper_dossier(dossier):
    """
    Pour chaque JPG dans dossier, génère une version flippée horizontalement
    dans un sous-dossier {dossier}_flip/ — ne retraite pas les fichiers déjà flippés.
    """
    dossier_flip = f"{dossier}_flip"
    os.makedirs(dossier_flip, exist_ok=True)

    images  = [f for f in os.listdir(dossier)
               if f.endswith(".jpg") and "_flip" not in f]
    nb_flip = 0
    for nom in images:
        chemin_orig = f"{dossier}/{nom}"
        chemin_flip = f"{dossier_flip}/{nom}"
        if os.path.exists(chemin_flip):
            continue
        img = Image.open(chemin_orig).convert("RGB")
        ImageOps.mirror(img).save(chemin_flip)
        nb_flip += 1
    return nb_flip

total_flip = 0
print("Data augmentation — flip horizontal :\n")

for d in sorted(os.listdir(DOSSIER_SEGMENTEES)):
    chemin = f"{DOSSIER_SEGMENTEES}/{d}"
    if not os.path.isdir(chemin):
        continue
    if "_flip" in d:
        continue  # ignorer les dossiers flip déjà créés

    if d == "cuivre_pdf":
        for edition in sorted(os.listdir(chemin)):
            sous = f"{chemin}/{edition}"
            if os.path.isdir(sous) and "_flip" not in edition:
                nb = flipper_dossier(sous)
                print(f"  {edition[:50]:50s} : +{nb} flips → {edition}_flip/")
                total_flip += nb
    else:
        nb = flipper_dossier(chemin)
        print(f"  {d:50s} : +{nb} flips → {d}_flip/")
        total_flip += nb

print(f"\n✓ {total_flip} images flippées générées")

Data augmentation — flip horizontal :

  bois_salomon_lyon1557                              : +192 flips → bois_salomon_lyon1557_flip/
  bois_solis_francfort1581                           : +187 flips → bois_solis_francfort1581_flip/
  bois_wickram_mayence1545                           : +52 flips → bois_wickram_mayence1545_flip/
  cuivre_clein_paris1637                             : +18 flips → cuivre_clein_paris1637_flip/
  cuivre_passe_metamorphoseon                        : +135 flips → cuivre_passe_metamorphoseon_flip/
  cuivre_passe_nasonis                               : +136 flips → cuivre_passe_nasonis_flip/
  cuivre_renouard_traduites                          : +16 flips → cuivre_renouard_traduites_flip/
  cuivre_renouard_traduittes                         : +17 flips → cuivre_renouard_traduittes_flip/

✓ 753 images flippées générées


## 8. Split du dataset train / val / test

Split stratifié 70 / 15 / 15 appliqué après augmentation.  
Les images originales et flippées d'une même source restent dans le même split.
On prend les images flippées que dans le / train.

In [2]:
import random
from PIL import Image
import re



def split_dataset_augmente(dossier_dataset, classe, ratio_train=0.7, ratio_val=0.15):
    """
    Split train/val/test en groupant chaque image originale avec son flip.
    Les flips sont identifiés par la présence de _flip_ dans le nom.
    Les flips vont uniquement dans train — val et test contiennent uniquement les originaux.
    """
    for split in ["train", "val", "test"]:
        os.makedirs(f"{dossier_dataset}/{split}/{classe}", exist_ok=True)

    toutes     = [f for f in os.listdir(f"{dossier_dataset}/{classe}") if f.endswith(".jpg")]
    originales = [f for f in toutes if "_flip_" not in f]
    flips      = [f for f in toutes if "_flip_" in f]

    random.shuffle(originales)

    n       = len(originales)
    n_train = int(n * ratio_train)
    n_val   = int(n * ratio_val)

    splits = {
        "train": originales[:n_train],
        "val"  : originales[n_train:n_train + n_val],
        "test" : originales[n_train + n_val:]
    }

    # Index des originaux train pour retrouver leurs flips
    originales_train = set(splits["train"])

    comptes = {"train": 0, "val": 0, "test": 0}

    for split, fichiers in splits.items():
        for f in fichiers:
            shutil.copy(f"{dossier_dataset}/{classe}/{f}",
                        f"{dossier_dataset}/{split}/{classe}/{f}")
            comptes[split] += 1

    # Ajouter les flips dans train — un flip appartient au même original
    # si son nom contient le nom de l'original sans _flip_
    for flip in flips:
        nom_original = flip.replace("_flip_", "_")
        if nom_original in originales_train:
            shutil.copy(f"{dossier_dataset}/{classe}/{flip}",
                        f"{dossier_dataset}/train/{classe}/{flip}")
            comptes["train"] += 1

    for split, n in comptes.items():
        print(f"  {split}/{classe} : {n} images")

        

        
# Nettoyer l'ancien dataset
for split in ["train", "val", "test"]:
    for classe in ["bois", "cuivre"]:
        dossier = f"{DOSSIER_DATASET}/{split}/{classe}"
        if os.path.exists(dossier):
            shutil.rmtree(dossier)

rassembler_illustrations(DOSSIER_SEGMENTEES, DOSSIER_DATASET)

print("\nSplit train/val/test avec augmentation :\n")
print("Classe bois :")
split_dataset_augmente(DOSSIER_DATASET, "bois")
print("\nClasse cuivre :")
split_dataset_augmente(DOSSIER_DATASET, "cuivre")

print("\n✓ Dataset final :")
for split in ["train", "val", "test"]:
    nb_b = len([f for f in os.listdir(f"{DOSSIER_DATASET}/{split}/bois")   if f.endswith(".jpg")])
    nb_c = len([f for f in os.listdir(f"{DOSSIER_DATASET}/{split}/cuivre") if f.endswith(".jpg")])
    print(f"  {split:5s} : {nb_b:3d} bois + {nb_c:3d} cuivre = {nb_b + nb_c} total")

✓ dataset/bois   : 862 images
✓ dataset/cuivre : 644 images

Split train/val/test avec augmentation :

Classe bois :
  train/bois : 602 images
  val/bois : 64 images
  test/bois : 66 images

Classe cuivre :
  train/cuivre : 450 images
  val/cuivre : 48 images
  test/cuivre : 49 images

✓ Dataset final :
  train : 602 bois + 450 cuivre = 1052 total
  val   :  64 bois +  48 cuivre = 112 total
  test  :  66 bois +  49 cuivre = 115 total
